# 01 — EDA & Data Preparation

Distribusi dataset, visualisasi sampel, pemeriksaan imbalance, dan pembuatan split stratified.

In [ ]:
# Google Colab Setup
# Jalankan sel ini PERTAMA di setiap session Colab baru.
import os, sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Sesuaikan jika nama folder di Google Drive berbeda
    PROJECT_PATH = '/content/drive/MyDrive/pcd-kelompok-17'
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, PROJECT_PATH)
    print('Setup selesai. Project:', PROJECT_PATH)
else:
    print('Lokal - skip mount Drive.')


In [ ]:
# Install dependencies - jalankan sekali per session Colab
%pip install -q opencv-python-headless scipy scikit-image scikit-learn statsmodels matplotlib seaborn pandas tqdm joblib kagglehub tensorflow


In [ ]:
import os
import sys
from pathlib import Path

_in_colab = "COLAB_RELEASE_TAG" in os.environ
ROOT = Path.cwd() if _in_colab else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.preprocessing import check_integrity
from src.utils import (
    KAGGLE_DATASET_SLUG,
    build_dataset_index,
    get_project_paths,
    make_splits,
    save_splits,
    set_seed,
)

set_seed(42)
paths = get_project_paths()
paths


In [ ]:
import shutil

import kagglehub

# True  = unduh dari Kaggle (default untuk Colab)
# False = pakai data yang sudah ada di data/raw/
DOWNLOAD_DATASET = True

if DOWNLOAD_DATASET:
    _in_colab = "COLAB_RELEASE_TAG" in os.environ
    if _in_colab:
        # Simpan kaggle.json di Google Drive: MyDrive/.kaggle/kaggle.json
        _kaggle_src = Path('/content/drive/MyDrive/.kaggle/kaggle.json')
        _kaggle_dst = Path('/root/.kaggle/kaggle.json')
        _kaggle_dst.parent.mkdir(exist_ok=True)
        shutil.copy2(_kaggle_src, _kaggle_dst)
        _kaggle_dst.chmod(0o600)
        print('Kaggle API key siap dari Drive')

    kaggle_path = Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG))
    print('Path to dataset:', kaggle_path)
    RAW_DATA_DIR = kaggle_path
else:
    RAW_DATA_DIR = paths['data_raw']
    print('Pakai data lokal:', RAW_DATA_DIR)


In [ ]:
df = build_dataset_index(RAW_DATA_DIR)
print(f"Total citra: {len(df)}")
print(f"Komoditas: {df['commodity'].nunique()}")
print(f"Label: {df['label'].value_counts().to_dict()}")
df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["label"].value_counts().plot(kind="bar", ax=axes[0], title="Distribusi Kelas")
df.groupby(["commodity", "label"]).size().unstack(fill_value=0).plot(
    kind="bar", stacked=True, ax=axes[1], title="Komoditas x Kelas"
)
plt.tight_layout()
plt.show()


In [ ]:
import cv2

sample = df.groupby(["commodity", "label"]).first().reset_index().head(6)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = cv2.imread(row["filepath"])
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{row['commodity']} -- {row['label']}")
    ax.axis("off")
plt.suptitle("Contoh Citra")
plt.tight_layout()
plt.show()


In [ ]:
corrupt = [fp for fp in df["filepath"] if not check_integrity(fp)]
print(f"Citra corrupt / tidak terbaca: {len(corrupt)}")


In [ ]:
train, val, test = make_splits(df)
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
split_path = save_splits(train, val, test, paths["splits"])
print(f"Split disimpan ke: {split_path}")
